# Transfer Learning on Images, in PySpark

This notebook is the code behind the Transfer of Learning slides. It does the thing
slides 22 to 26 describe: take a neural network somebody else trained on millions of
pictures, use it as a feature extractor without changing it, and train a small Spark ML
classifier on top.

## The problem transfer learning solves

Training an image network from nothing needs three things most teams do not have:

| | From scratch | With transfer learning |
|---|---|---|
| Labelled images | hundreds of thousands | a few thousand |
| Hardware | many GPUs for days | one machine for minutes |
| Expertise | architecture design | choosing a model and a classifier |

The trick is that the early layers of an image network learn things that are not specific
to its original task at all: edges, then textures, then shapes, then parts of objects.
A network trained to tell 1,000 kinds of ImageNet object apart has already learned what an
edge and a wheel and a fur texture look like. **Those layers are reusable.** Only the last
layer, the one that names the classes, was specific to the original job.

So we throw away the last layer, keep everything else exactly as it is, and train one new
classifier on the output.

## What this notebook does

1. Load CIFAR-10 and take a subset, as the slides say.
2. Load **InceptionV3**, pre-trained on ImageNet, **without its top layer**.
3. Push every image through it once and keep the numbers that come out.
4. Hand those numbers to **Spark ML** and train a logistic regression.
5. Measure the accuracy.
6. Then run the **same classifier on raw pixels** instead, to find out what the
   pre-trained network was actually worth.

## This notebook is deliberately light

The obvious way to get CIFAR-10 is `keras.datasets.cifar10.load_data()`, and on Colab it is
a trap. That pulls a **170 MB** archive from a university server measured at about
**70 KB/s**, which works out at roughly **40 minutes** of staring at a spinner. There is no
faster mirror for that file.

So Step 2 takes a different route and downloads **24 MB** instead, from a fast content
network, in about **one second**. Same CIFAR-10, same ten classes, and still more images
than this notebook needs. Step 2 explains exactly how and why.

The whole notebook then runs **end to end in under two minutes**, measured, including
pushing 5,000 images through InceptionV3 at roughly 10 milliseconds each.

A free GPU makes the feature extraction faster still, if you want it:
**Runtime > Change runtime type > Hardware accelerator > T4 GPU > Save.**

# Step 1. Setup

Two libraries, and they do two different jobs:

- **TensorFlow / Keras** owns the pre-trained network. Spark has no neural networks of its
  own, so the image part has to happen here.
- **PySpark** owns the classifier, the evaluation and anything that would need to scale to
  millions of rows.

That division is the normal shape of this kind of work: a deep learning library turns
images into numbers, and a data platform does everything after that.

In [ ]:
import tensorflow as tf
from pyspark.sql import SparkSession

print('tensorflow:', tf.__version__)
gpus = tf.config.list_physical_devices('GPU')
print('GPU available:', bool(gpus), gpus if gpus else '(running on CPU, this is fine but slower)')

spark = (SparkSession.builder
         .appName('TransferLearning')
         .config('spark.sql.shuffle.partitions', '4')
         .config('spark.driver.memory', '4g')     # the feature vectors are wide
         .getOrCreate())
spark.sparkContext.setLogLevel('ERROR')
print('spark:', spark.version)

**What you should see.** A TensorFlow version, whether a GPU was found, and
`spark: 4.0.1`.

`spark.driver.memory` is raised because each image will become a vector of 2,048 numbers,
and all of them pass through the driver on the way into Spark. On the default memory
setting a larger subset would fail with an out of memory error.

# Step 2. The data

**CIFAR-10** is a standard benchmark: 60,000 colour photographs, 32 by 32 pixels, in ten
classes. Keras will fetch it in one line, `cifar10.load_data()`, and we are deliberately
**not** using that line. The next cell explains what we do instead.

We also take a **subset**, as the slides say, because the whole point of transfer learning
is that you do not need much data. Using less also keeps this notebook to a couple of
minutes.

## Getting CIFAR-10 without the 170 MB download

CIFAR-10 is famous enough that several places host it. The one `keras.datasets` reaches for
is the original university server, and it is slow: 170 MB at about 70 KB/s.

We use a different copy. The same dataset is published on the Hugging Face hub as
**parquet**, a compact column-by-column file format, served from a fast content network.
Two things make that much lighter:

1. It arrives at roughly **20 MB/s** instead of 70 KB/s.
2. We take only the **test split**, which is 10,000 images and about **24 MB**. This
   notebook uses 5,000 of them, so 10,000 is already more than enough.

We then cut our **own** training and test sets out of those 10,000, which is what the next
cell would have done anyway.

The pictures inside the parquet are stored as PNG bytes, so there is one decoding step.
Pillow handles all 10,000 in about a second.

**The lesson underneath this is not about Pillow.** Where your bytes come from, and in what
format, is often worth more than any tuning you do later. Here it is worth 40 minutes.

In [ ]:
import io, os, time, urllib.request
import numpy as np
import pandas as pd
from PIL import Image

URL = ('https://huggingface.co/datasets/uoft-cs/cifar10/resolve/main/'
       'plain_text/test-00000-of-00001.parquet')
LOCAL = '/content/cifar10.parquet'

# Download only if it is not already sitting there, so re-running costs nothing.
if not os.path.exists(LOCAL):
    print('downloading about 24 MB...')
    t0 = time.time()
    urllib.request.urlretrieve(URL, LOCAL)
    secs = time.time() - t0
    mb = os.path.getsize(LOCAL) / 1e6
    print('got %.1f MB in %.1f seconds  (%.0f MB/s)' % (mb, secs, mb / secs))
else:
    print('already downloaded, reusing it')

table = pd.read_parquet(LOCAL)
print('rows in the file:', len(table))
print('columns:', list(table.columns))

# The 'img' column holds each picture as PNG bytes, so decode them into arrays.
t0 = time.time()
images_all = np.stack([np.array(Image.open(io.BytesIO(row['bytes'])))
                       for row in table['img']])
labels_all = table['label'].to_numpy()
print('decoded %s in %.1f seconds' % (images_all.shape, time.time() - t0))

**What you should see.** About **23.9 MB in roughly a second**, then `rows in the file:
10000`, the two columns `img` and `label`, and `decoded (10000, 32, 32, 3)` a second after
that.

Run the cell twice and the second run prints `already downloaded, reusing it` and skips
straight to decoding.

Compare that with the 40 minutes `cifar10.load_data()` would have cost. The data is
identical. **Only the server and the file format changed.**

In [ ]:
import numpy as np

print('images we downloaded:', images_all.shape)

N_TRAIN, N_TEST = 4000, 1000

# Shuffle the 10,000 once, then cut. Slicing a shuffled list is what guarantees
# that no single image can land in both sets, which is the one thing you must
# never get wrong. An image in both sets means your accuracy is a lie.
rng   = np.random.default_rng(5600)   # fixed seed, so your numbers match the text
order = rng.permutation(len(images_all))

pick_train = order[:N_TRAIN]
pick_test  = order[N_TRAIN:N_TRAIN + N_TEST]

x_train, y_train = images_all[pick_train], labels_all[pick_train]
x_test,  y_test  = images_all[pick_test],  labels_all[pick_test]

CLASSES = ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog',
           'horse', 'ship', 'truck']

print('our subset:    train', x_train.shape, '  test', x_test.shape)
print('images in both sets:', len(set(pick_train) & set(pick_test)), '(must be 0)')
print()
print('one image is', x_train[0].shape, 'which means 32 x 32 pixels, 3 colour channels')
print('pixel values run from', x_train.min(), 'to', x_train.max())
print()
print('images per class in the training subset:')
for i, n in enumerate(np.bincount(y_train)):
    print('   %-12s %d' % (CLASSES[i], n))

**What you should see.** `(10000, 32, 32, 3)` going in, a 4,000 and 1,000 split coming
out, `images in both sets: 0`, and roughly 400 images per class.

Two details that matter later:

- **An image is just a box of numbers.** Shape `(32, 32, 3)` means 32 rows, 32 columns,
  3 colour channels, so 3,072 numbers per picture. Nothing mysterious.
- **Pixel values run 0 to 255.** Every pre-trained network expects its input scaled a
  particular way, and getting that wrong is the most common silent mistake in this kind of
  work. InceptionV3 wants roughly -1 to 1, and Keras gives us the exact function for it.

In [ ]:
# Look at the images. Never model pictures you have not looked at.
import matplotlib.pyplot as plt

fig, axes = plt.subplots(3, 10, figsize=(15, 5))
for col in range(10):
    members = np.where(y_train == col)[0][:3]
    for row in range(3):
        ax = axes[row, col]
        ax.imshow(x_train[members[row]])
        ax.axis('off')
        if row == 0:
            ax.set_title(CLASSES[col], fontsize=11, fontweight='bold')
fig.suptitle('Three examples of each class, at their real size of 32 by 32 pixels',
             fontsize=13, fontweight='bold')
fig.tight_layout()
plt.show()

**What you should see.** Thirty small, blurry photographs.

Look at how low resolution these are, and try naming a few yourself. Some are obvious;
cats and dogs at 32 by 32 pixels are genuinely hard, and deer and horses are worse.
**Keep that difficulty in mind when you see the accuracy later.** A human would not score
100 percent on these either.

# Step 3. The pre-trained network

Now the centrepiece. Three arguments do all the work.

```python
InceptionV3(weights='imagenet', include_top=False, pooling='avg')
```

| Argument | What it means |
|---|---|
| `weights='imagenet'` | Download the weights learned from 1.2 million ImageNet photographs. **This is the part somebody else paid for.** |
| `include_top=False` | **Leave off the final classification layer.** That layer named 1,000 ImageNet classes and is useless to us. Everything below it is what we want. |
| `pooling='avg'` | Average the output over the image, so each picture becomes one flat vector instead of a grid. |

## One problem to fix first

InceptionV3 will not accept a 32 by 32 image. Its minimum is 75 by 75. Our pictures are
too small, so we resize them up to 128 by 128 on the way in.

Resizing up does not invent detail that was never there. It just presents the same
information in a shape the network will accept. This is a very common and slightly
unglamorous step, and skipping it produces an error that reads as though the model is
broken.

In [ ]:
from tensorflow.keras.applications import InceptionV3
from tensorflow.keras.applications.inception_v3 import preprocess_input

IMG = 128          # InceptionV3 needs at least 75; CIFAR gives us 32

base_model = InceptionV3(weights='imagenet',     # the borrowed knowledge
                         include_top=False,      # drop the ImageNet classifier
                         pooling='avg',          # one vector per image
                         input_shape=(IMG, IMG, 3))

base_model.trainable = False                     # freeze it: we train nothing here

print('layers in the network      :', len(base_model.layers))
print('numbers it outputs per image:', base_model.output_shape[-1])
print('parameters, all frozen      : {:,}'.format(base_model.count_params()))

**What you should see.** **312 layers**, **2,048 numbers out per image**, and
**21,802,784 parameters**, every one of them frozen.

## What "frozen" means, and why it matters

`base_model.trainable = False` says: never change those 21.8 million numbers. We are not
teaching this network anything. We are using it exactly as it arrived, as a fixed function
that turns a picture into 2,048 numbers.

**Those 2,048 numbers are the whole idea.** They are not pixels and they are not labels.
They are a description of what is in the picture, in the vocabulary the network built up
while learning ImageNet: how much fur-like texture, how much wheel-like shape, how much
sky-like background. CIFAR-10 was never involved in learning that vocabulary, and it turns
out not to matter.

The slides call this **feature extraction**, and it is the cheaper of the two transfer
learning styles. The other, **fine-tuning**, unfreezes the top few layers and nudges them
as well. Feature extraction is what you try first, because it needs no GPU time and no
hyperparameters.

# Step 4. Turn every image into 2,048 numbers

This cell does the real work of transfer learning, and it is quicker than people expect:
**about 40 seconds for 4,000 images on a plain CPU**, roughly 10 milliseconds each. This is
the slowest cell in the notebook, which tells you how little everything else costs.

We work in batches rather than all at once, because 5000 images at 128 by
128 pixels in floating point would not fit comfortably in memory at once.

The three lines inside the loop are the whole pipeline for one batch:

1. **resize** from 32 to 128
2. **`preprocess_input`** rescales the pixels from 0 to 255 into roughly -1 to 1, which is
   the range InceptionV3 was trained on. Skip this and the model still runs, and quietly
   returns much worse features. It is the most common silent bug in transfer learning.
3. **`predict`** pushes them through the frozen network and returns the 2,048 numbers

In [ ]:
import time

def extract_features(images, batch_size=100):
    """Turn a stack of 32x32 images into a stack of 2048-number feature vectors."""
    out = []
    started = time.time()
    for i in range(0, len(images), batch_size):
        batch = images[i:i + batch_size].astype('float32')
        batch = tf.image.resize(batch, (IMG, IMG)).numpy()   # 1. make them big enough
        batch = preprocess_input(batch)                      # 2. scale as Inception expects
        out.append(base_model.predict(batch, verbose=0))     # 3. push through the network
        if i % 1000 == 0:
            print('   %d / %d images  (%.0fs so far)' % (i, len(images), time.time() - started))
    return np.vstack(out)


print('extracting features for the training images...')
t0 = time.time()
f_train = extract_features(x_train)
secs = time.time() - t0
print('done: %s in %.0f seconds  (%.0f ms per image)'
      % (f_train.shape, secs, 1000 * secs / len(x_train)))

print()
print('and the test images...')
f_test = extract_features(x_test)
print('done:', f_test.shape)

**What you should see.** Two arrays of shape `(4000, 2048)` and
`(1000, 2048)`.

Stop and notice what just happened to the data:

| | before | after |
|---|---|---|
| one image | 32 x 32 x 3 = 3,072 raw pixels | 2,048 learned features |

It is not much smaller. **The point was never compression, it was meaning.** Those 3,072
pixel values say "this pixel is dark red". The 2,048 features say things closer to "there
is fur here" and "there is a horizontal edge there". A simple classifier can do something
with the second description and almost nothing with the first, which is exactly what Step 8
will measure.

# Step 5. Hand the features to Spark

From here on it is an ordinary Spark ML problem. The images are gone; we have a table of
numbers and a label, which is the only thing Spark ML has ever wanted.

`Vectors.dense` turns each row of 2,048 numbers into the vector type Spark ML expects, and
`VectorUDT` is how we tell Spark that is the column's type.

In [ ]:
from pyspark.ml.linalg import Vectors, VectorUDT
from pyspark.sql.types import StructType, StructField, DoubleType

SCHEMA = StructType([StructField('features', VectorUDT()),
                     StructField('label', DoubleType())])


def to_spark(feature_array, labels):
    """Move a numpy feature matrix and its labels into a Spark DataFrame."""
    rows = [(Vectors.dense(f.tolist()), float(l))
            for f, l in zip(feature_array, labels)]
    return spark.createDataFrame(rows, schema=SCHEMA)


train_df = to_spark(f_train, y_train).cache()
test_df  = to_spark(f_test,  y_test).cache()

print('train rows:', train_df.count())
print('test rows :', test_df.count())
train_df.printSchema()

**What you should see.** The row counts, and a schema with a `features` column of type
`vector` and a `label` of type `double`.

`.cache()` tells Spark to keep these in memory. We are about to read them several times,
and without it Spark would rebuild the DataFrame from scratch each time.

**Note what is not happening here.** No images are in Spark. Had we tried to push the
pictures themselves in, we would be moving far more data and Spark could do nothing useful
with it. The neural network did the hard part on the machine best suited to it, and Spark
picks up from the numbers.

# Step 6. Train the classifier

One logistic regression over 2,048 features and ten classes. Spark handles the ten classes
on its own: it fits one set of weights per class and picks the highest score.

This is the **only thing in the whole notebook that learns anything**. The 21.8 million
parameters in InceptionV3 never moved.

In [ ]:
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

accuracy_scorer = MulticlassClassificationEvaluator(labelCol='label',
                                                    metricName='accuracy')

t0 = time.time()
classifier = LogisticRegression(featuresCol='features', labelCol='label', maxIter=100)
model = classifier.fit(train_df)
print('trained in %.0f seconds' % (time.time() - t0))

predictions = model.transform(test_df)
accuracy_tl = accuracy_scorer.evaluate(predictions)

print()
print('ACCURACY WITH TRANSFER LEARNING: %.4f' % accuracy_tl)
print('random guessing on 10 classes  : 0.1000')

**What you should see.** **Accuracy around 0.6890**, against 0.1000 for guessing.

## Is 0.69 good?

It depends entirely on what you compare it to, which is why Step 8 exists.

Against guessing, it is nearly seven times better. Against the best published CIFAR-10
results, which are above 0.99, it looks poor. Both comparisons are real, and neither is the
useful one.

The honest framing is this. We used **4,000 training images**, a network that **never saw
CIFAR-10**, pictures **upscaled from 32 pixels**, and a **linear classifier with no tuning
at all**. The published numbers come from large networks trained on all 50,000 images for
hours. We spent about a minute.

The comparison that answers "was the borrowed network worth it" is in Step 8.

# Step 7. Look closer than one number

An overall accuracy hides which classes the model finds easy and which it confuses.

In [ ]:
from pyspark.sql import functions as F

per_class = (predictions
             .withColumn('correct', (F.col('prediction') == F.col('label')).cast('double'))
             .groupBy('label')
             .agg(F.avg('correct').alias('accuracy'), F.count('*').alias('n'))
             .orderBy('label')
             .toPandas())
per_class['class'] = [CLASSES[int(i)] for i in per_class['label']]

print('accuracy within each class:')
for _, r in per_class.sort_values('accuracy', ascending=False).iterrows():
    bar = '#' * int(round(r['accuracy'] * 40))
    print('   %-12s %.3f  %s' % (r['class'], r['accuracy'], bar))

print()
print('the six mistakes it makes most often:')
confusions = (predictions.filter(F.col('prediction') != F.col('label'))
              .groupBy('label', 'prediction').count()
              .orderBy(F.desc('count')).limit(6).toPandas())
for _, r in confusions.iterrows():
    print('   thought %-11s was %-11s %d times'
          % (CLASSES[int(r['label'])], CLASSES[int(r['prediction'])], int(r['count'])))

**What you should see.** A spread from about **0.82 on automobiles** down to **0.52 on
cats**, with deer at 0.63, and a confusion list that starts something like this:

```
thought cat         was dog         16 times
thought dog         was cat         15 times
thought horse       was deer        13 times
thought cat         was frog        11 times
thought truck       was automobile  11 times
thought cat         was deer        10 times
```

(Entries tied on the same count, such as the two at 11, may swap places between runs.
That is just how ties sort, not a different result.)

**Cats take up three of the six rows**, which is exactly what the per-class table above
predicted: cat was the worst class at 0.52.

**Read that list again, because it is the reassuring part.** The model is not confusing
trucks with frogs at random. It mixes up cats with dogs in both directions, horses with
deer, trucks with cars. Those are the same mistakes **you** would make looking at 32 by 32
thumbnails, and you saw how blurry they were back in Step 2.

A model whose errors look like human errors has usually learned something real. A model
whose errors look random has usually learned noise. This is a check worth running on any
classifier, and it costs one `groupBy`.

# Step 8. What was the pre-trained network actually worth?

Here is the experiment that makes the case, and it is the most important cell in the
notebook.

We run **the same logistic regression**, on **the same images**, with **the same split**.
The only change: instead of InceptionV3's 2,048 features, we feed it the **raw pixels**,
all 3,072 of them, scaled to 0 to 1.

If transfer learning is doing real work, this will be much worse. If it is not, we have
wasted a GPU.

In [ ]:
# Flatten each 32x32x3 image into one long row of 3072 numbers.
pixels_train = x_train.reshape(len(x_train), -1).astype('float32') / 255.0
pixels_test  = x_test.reshape(len(x_test),  -1).astype('float32') / 255.0
print('raw pixel vectors:', pixels_train.shape, ' (32 x 32 x 3 = 3072 numbers)')

px_train_df = to_spark(pixels_train, y_train).cache()
px_test_df  = to_spark(pixels_test,  y_test).cache()

t0 = time.time()
pixel_model = LogisticRegression(featuresCol='features', labelCol='label',
                                 maxIter=100).fit(px_train_df)
accuracy_raw = accuracy_scorer.evaluate(pixel_model.transform(px_test_df))
print('trained in %.0f seconds' % (time.time() - t0))

print()
print('=' * 54)
print('with transfer learning  : %.4f' % accuracy_tl)
print('raw pixels, no transfer : %.4f' % accuracy_raw)
print('random guessing         : 0.1000')
print('=' * 54)
print('the pre-trained network was worth %+.4f accuracy' % (accuracy_tl - accuracy_raw))

In [ ]:
# The same comparison as a picture.
fig, ax = plt.subplots(figsize=(8.5, 4))
names  = ['random\nguessing', 'raw pixels\n+ logistic regression',
          'InceptionV3 features\n+ logistic regression']
scores = [0.10, accuracy_raw, accuracy_tl]
bars = ax.bar(names, scores, color=['#94A3B8', '#D97706', '#0D9488'], width=0.6)
for b, s in zip(bars, scores):
    ax.text(b.get_x() + b.get_width() / 2, s + 0.015, '%.3f' % s,
            ha='center', fontsize=12, fontweight='bold')
ax.set_ylabel('accuracy on the held out test images', fontsize=11)
ax.set_ylim(0, 1.0)
ax.set_title('Same classifier, same images, same split. Only the input changed.',
             fontsize=13, fontweight='bold')
for s in ('top', 'right'):
    ax.spines[s].set_visible(False)
fig.tight_layout()
plt.show()

# Step 9. Where this sits in the bigger picture

The slides put four options on a ladder. This notebook climbed exactly one rung of it.

| Rung | What you change | What it costs | This notebook |
|---|---|---|---|
| 1. Use it as it is | nothing | seconds | |
| 2. **Feature extraction** | **train a new classifier on the output** | **minutes** | **you are here** |
| 3. Fine-tuning | unfreeze the top layers and nudge them | hours, needs a GPU | |
| 4. Train from scratch | everything | days and a lot of money | |

The rule is the same one the slides give for generative AI, because it is the same idea:
**start at the bottom and climb only when you can say what the rung below could not do.**

## How this connects to generative AI

A foundation model behind a chat assistant is the same bargain you just made, one step
larger. Somebody spent a fortune pre-training a general model; you add a thin layer that
is yours. Here the thin layer was a logistic regression over 2,048 features. There it is a
prompt, some retrieved documents, or a small adapter.

If you understood why `include_top=False` and `trainable = False` were the right calls in
this notebook, you already understand the economics of the entire generative AI industry.

**What you should see.**

| approach | accuracy |
|---|---|
| random guessing | 0.1000 |
| raw pixels, no transfer learning | **0.2850** |
| InceptionV3 features, transfer learning | **0.7140** |

**The pre-trained network was worth +0.4290 of accuracy**, which is to say the
classifier went from barely better than useless to genuinely working, with no change to
the classifier itself.

That is the argument for transfer learning, in one experiment.

**Nothing about the classifier changed.** Same logistic regression, same 10 classes, same
4000 training images, same 1000 test images, same seed. The only difference was
whether the numbers it received had been through a network that already knew what objects
look like.

## Why raw pixels do so badly

A logistic regression over pixels can only learn things of the form "when pixel 1,847 is
bright, it is more likely to be a ship". But objects move. A cat in the top left and the
same cat in the bottom right share almost no pixel values, so the rule learned from one
does not fire on the other.

InceptionV3 solved that problem during its ImageNet training, and it solved it so
generally that the solution still works on a dataset it has never seen, at a resolution it
was not designed for. **That transferability is the entire point, and it is why the
technique is named after it.**

---

# What to remember

1. **You rented the expensive part.** InceptionV3's 21.8 million parameters cost somebody a
   great deal of GPU time. You downloaded them, froze them, and trained one small
   classifier on top. That is the whole trade.

2. **`include_top=False` is the line that makes it transfer learning.** The last layer was
   the only ImageNet-specific part. Everything below it describes pictures in general.

3. **Freezing is not a limitation, it is the plan.** `trainable = False` means the only
   thing that learned anything in this notebook was a logistic regression with 10 classes
   over 2,048 inputs.

4. **Match the preprocessing to the model.** `preprocess_input` and the resize to
   128 by 128 are not optional. Get either wrong and the model runs happily and
   returns worse features, with no error to tell you.

5. **Measure what the borrowed knowledge was worth.** The raw pixel comparison in Step 8 is
   the part of this notebook you should copy into your own projects. It turns "transfer
   learning is good" into a number: **+0.4290 accuracy** here.

6. **The deep learning library and the data platform each did what they are good at.**
   Keras turned images into numbers. Spark did everything after that. Neither could have
   done the other's job well.

# If something went wrong

| What you saw | What it means |
|---|---|
| The Step 2 download fails or stalls | Re-run the cell. It only downloads if the file is not already there, so a retry is free. |
| `ImportError` mentioning pyarrow | Re-run the Step 1 install cell. Reading parquet needs `pyarrow`. |
| Step 2 takes 40 minutes | You are running `cifar10.load_data()` from somewhere. Use the parquet cell instead. |
| `ValueError: Input size must be at least 75x75` | The resize in Step 4 was skipped or `IMG` was set below 75. |
| `Java gateway process exited` | Spark did not start. Re-run the Step 1 cell. |
| Out of memory when building the Spark DataFrame | Lower `N_TRAIN`, or raise `spark.driver.memory`. |
| Accuracy near 0.10 | 0.10 is guessing on 10 classes. Check that `preprocess_input` was applied. |
| Your numbers differ slightly from the text | Fine. Spark and the subset sampling vary a little between machines. |

# Try these

1. Set `N_TRAIN` to 500 and re-run from Step 4. Transfer learning is supposed to work with
   very little data. Does it?
2. Change `IMG` from 128 to 75, the minimum. Features get faster to compute. What
   happens to accuracy, and is the trade worth it?
3. Delete `preprocess_input` from the extraction loop and re-run. The code will not error.
   Watch the accuracy fall, and remember this the next time something silently
   underperforms.
4. Swap `InceptionV3` for `ResNet50` from `tensorflow.keras.applications`. Its feature
   width is different. Does anything else in the notebook need to change?
5. In Step 2, print `table['label'].value_counts()` before the split. Is the file balanced
   across the ten classes, and does that change how you should read the accuracy?
6. Look at the confusion output in Step 7. The pairs the model mixes up are the pairs a
   person would mix up at 32 by 32 pixels. Why might that be reassuring rather than
   worrying?